# EpidemicWatch — Phase 4: Federated Learning (Flower + LoRA + FedAvg/FedProx)

Federates the Phase 3 centralized baseline (`Bio_ClinicalBERT` + LoRA) across your
3 simulated district hospitals using the **Flower** framework. Only LoRA adapter +
classification-head weights are ever transmitted between clients and server — the
frozen Bio_ClinicalBERT base model never moves, which is the whole point of doing
LoRA-federation instead of full-model federation.

**Data:** Phase 2's Kaggle *Diseases and Symptoms* data (~15,000 template-generated
sentences, 5 syndrome categories), split across 3 hospitals (~10.5k training rows in total).
As in Phase 3, results are a **synthetic-text benchmark**, not real-world performance.

**Centralized baselines from Phase 3 (pooled non-IID test set):**
- No augmentation (the like-for-like comparison, since federated clients below do not augment):
  91.4% accuracy, 0.913 macro-F1
- Augmented run (best centralized variant): 93.4% accuracy, 0.933 macro-F1

**Experiments run in this notebook, all under the same conditions otherwise:**
1. FedAvg on the **non-IID** hospital split (μ=0)
2. FedProx on the **non-IID** split, μ=0.01
3. FedProx on the **non-IID** split, μ=0.1
4. FedAvg on the **IID** split (control, should converge fastest/highest)

> **A note on the Flower API:** this notebook uses `flwr.simulation.run_simulation`
> with the `ClientApp`/`ServerApp` pattern. Flower's docs currently recommend the
> `flwr run` CLI workflow instead, but that requires a multi-file project
> structure (`pyproject.toml` etc.) that doesn't fit a single Colab notebook.
> `run_simulation` still works correctly (just prints a deprecation warning you
> can ignore) and is the standard way to run Flower simulations from a notebook.

> Run this in Google Colab with a **GPU runtime**. Clients share the GPU through
> Ray (`CLIENT_NUM_GPUS` below). With ~10k training rows, CPU-only clients would take
> many hours, so GPU sharing is on by default. If a Ray/GPU error appears, set
> `CLIENT_NUM_GPUS = 0.0` and lower `NUM_ROUNDS` for a quick check.

## 0. Setup

In [ ]:
!pip install -q -U peft transformers accelerate flwr ray
# Colab pre-installs an old torchao (quantization library) that peft checks for
# and hard-fails on if present-but-too-old, even though we never use it here.
# Removing it entirely avoids the version-check crash.
!pip uninstall -y -q torchao 2>/dev/null || true

import os
import json
import random
import time
import zipfile
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import seaborn as sns

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from peft.utils import get_peft_model_state_dict, set_peft_model_state_dict

from flwr.client import ClientApp, NumPyClient
from flwr.common import Context, ndarrays_to_parameters
from flwr.server import ServerApp, ServerAppComponents, ServerConfig
from flwr.server.strategy import FedAvg
from flwr.simulation import run_simulation

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ---- Config ---------------------------------------------------------------
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
DATA_DIR = "epidemicwatch_data"
MAX_LENGTH = 96           # same as Phase 3
BATCH_SIZE = 32           # same as Phase 3
LOCAL_LR = 1e-4
LOCAL_EPOCHS = 1          # local epochs per client per round
NUM_ROUNDS = 6            # 6 rounds x 1 local epoch = same epoch budget as Phase 3 (6 epochs)
NUM_HOSPITALS = 3
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1
CLASSIFIER_DROPOUT = 0.3
# From Phase 3. Federated clients here do NOT augment, so the no-augment run is the like-for-like baseline.
CENTRALIZED_BASELINE = {"accuracy": 0.9139, "f1_macro": 0.9131}            # Phase 3, no augmentation
CENTRALIZED_BASELINE_AUG = {"accuracy": 0.9338, "f1_macro": 0.9331}        # Phase 3, augmented
OUTPUT_DIR = "phase4_federated_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Client-side compute: 3 Ray actors share one GPU (0.33 each). A T4 has enough
# memory for this (Phase 3 peaked at ~2.5 GB per model). If Ray complains about
# GPUs, set CLIENT_NUM_GPUS = 0.0 (CPU clients; very slow with this data size).
CLIENT_NUM_GPUS = 0.33

# "inprocess": clients train one after another in this notebook process on the GPU, using the same
#              HospitalClient code and FedAvg weighting. Reliable on Colab (default).
# "flower":    Flower simulation with Ray actors. On some Colab setups the Ray GPU workers crash
#              (SIGFPE in dropout_cuda) and every round returns 0 results.
SIMULATION_BACKEND = "inprocess"
CLIENT_RESOURCES = {"num_cpus": 1, "num_gpus": CLIENT_NUM_GPUS}

print("Config set.")

## 1. Load Data (hospital shards + pooled val/test) and Tokenizer

Loads Phase 2's non-IID and IID hospital shards, plus builds pooled val/test
sets for server-side round-by-round evaluation and final test-set comparison.
The non-IID pooled sets are identical to Phase 3's. IID runs are evaluated on the
IID pooled val/test sets, because the IID and non-IID shardings put different rows
in their test sets: evaluating an IID-trained model on the non-IID test set could
mean testing on rows it trained on.

In [ ]:
def data_root_is_valid(root):
    # True only if root actually contains the expected hospital shard files,
    # not just a folder with the right name.
    probe = os.path.join(root, "non_iid", "hospital_0", "train.csv")
    return os.path.isfile(probe)

def find_data_root(start=".", max_depth=3):
    # Search a few levels deep for a directory that actually has the expected
    # structure -- handles the common case where a zip was extracted with an
    # extra nesting level (e.g. epidemicwatch_data/epidemicwatch_data/...).
    import glob
    for depth in range(max_depth + 1):
        pattern = os.path.join(start, *(["*"] * depth), "non_iid", "hospital_0", "train.csv")
        matches = glob.glob(pattern)
        if matches:
            return os.path.dirname(os.path.dirname(os.path.dirname(matches[0])))
    return None

resolved_root = find_data_root()

if resolved_root is None:
    from google.colab import files
    print(f"Could not find a valid '{DATA_DIR}' structure anywhere in the current "
          f"directory tree. Please upload epidemicwatch_data.zip from Phase 2.")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    extract_target = DATA_DIR
    with zipfile.ZipFile(zip_name, "r") as z:
        z.extractall(extract_target)
    resolved_root = find_data_root()
    if resolved_root is None:
        print("Still couldn't locate the expected structure after extraction. Contents found:")
        for dirpath, dirnames, filenames in os.walk(extract_target):
            depth = dirpath.replace(extract_target, "").count(os.sep)
            if depth <= 3:
                print("  " * depth + os.path.basename(dirpath) + "/")
        raise FileNotFoundError(
            "Expected 'non_iid/hospital_0/train.csv' somewhere under the uploaded zip "
            "but didn't find it -- check the printed structure above and adjust DATA_DIR manually."
        )

DATA_DIR = resolved_root
print(f"Using data root: '{DATA_DIR}/' (validated: hospital shard files found).")

In [ ]:
def load_hospital_split(hospital_idx, split_name, source, data_dir=DATA_DIR):
    path = os.path.join(data_dir, source, f"hospital_{hospital_idx}", f"{split_name}.csv")
    return pd.read_csv(path)

def load_pooled_split(split_name, source, data_dir=DATA_DIR):
    dfs = [load_hospital_split(h, split_name, source, data_dir) for h in range(NUM_HOSPITALS)]
    return pd.concat(dfs, ignore_index=True)

# Categories / label maps -- built from the pooled non-IID train set (covers all classes)
_ref_df = load_pooled_split("train", "non_iid")
CATEGORIES = sorted(_ref_df["syndrome_category"].unique())
label2id = {c: i for i, c in enumerate(CATEGORIES)}
id2label = {i: c for c, i in label2id.items()}
NUM_LABELS = len(CATEGORIES)
print("Categories:", CATEGORIES)

# Pooled val (round-by-round eval) and test (final comparison) sets, one pair per data source
pooled_sets = {}
for source in ("non_iid", "iid"):
    v, t = load_pooled_split("val", source), load_pooled_split("test", source)
    for d in (v, t):
        d["label_id"] = d["syndrome_category"].map(label2id)
        assert d["label_id"].notna().all()
    pooled_sets[source] = (v, t)
    print(f"{source}: pooled val {len(v)} | pooled test {len(t)}")

# Hospital train sizes (sanity check, and shows how skewed the non-IID split is)
for source in ("non_iid", "iid"):
    sizes = [len(load_hospital_split(h, "train", source)) for h in range(NUM_HOSPITALS)]
    print(f"{source} train sizes per hospital: {sizes} (total {sum(sizes)})")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SymptomDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=MAX_LENGTH):
        self.texts = df["text"].tolist()
        self.labels = df["label_id"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding="max_length",
                              max_length=self.max_length, return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

pooled_loaders = {
    source: (DataLoader(SymptomDataset(v, tokenizer), batch_size=BATCH_SIZE),
             DataLoader(SymptomDataset(t, tokenizer), batch_size=BATCH_SIZE))
    for source, (v, t) in pooled_sets.items()
}
print("Data ready.")

## 2. Model + LoRA \<-\> Flower Parameter Conversion Helpers

In [ ]:
def build_lora_model():
    base_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id,
        classifier_dropout=CLASSIFIER_DROPOUT,
    )
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=["query", "value"], modules_to_save=["classifier"],
    )
    return get_peft_model(base_model, lora_config)


# Fixed key order for LoRA state dict -- confirmed identical across fresh model
# instances of the same architecture, which is required since Flower passes
# plain lists of ndarrays (not named dicts) between client and server.
_REFERENCE_KEYS = list(get_peft_model_state_dict(build_lora_model()).keys())
print(f"LoRA + classifier trainable tensors transmitted per round: {len(_REFERENCE_KEYS)}")

def get_lora_ndarrays(model):
    sd = get_peft_model_state_dict(model)
    return [sd[k].cpu().numpy() for k in _REFERENCE_KEYS]

def set_lora_ndarrays(model, ndarrays):
    sd = OrderedDict((k, torch.tensor(v)) for k, v in zip(_REFERENCE_KEYS, ndarrays))
    set_peft_model_state_dict(model, sd)

def transmitted_size_mb(ndarrays):
    return sum(arr.nbytes for arr in ndarrays) / 1e6

print(f"Approx. size transmitted per round per client: "
      f"{transmitted_size_mb(get_lora_ndarrays(build_lora_model())):.2f} MB")


def get_prox_params(model):
    """Live (grad-enabled) trainable parameters, in the same order as _REFERENCE_KEYS.
    Needed for FedProx: get_peft_model_state_dict() returns DETACHED tensors, so a
    proximal term built from it would have zero gradient and FedProx would silently
    behave exactly like FedAvg."""
    live = {n.replace(".modules_to_save", "").replace(".default", ""): p
            for n, p in model.named_parameters() if p.requires_grad}
    missing = [k for k in _REFERENCE_KEYS if k not in live]
    assert not missing, f"Could not map {len(missing)} LoRA keys to live params, e.g. {missing[:2]}"
    return [live[k] for k in _REFERENCE_KEYS]

_test_model = build_lora_model()
assert all(p.requires_grad for p in get_prox_params(_test_model))
print(f"FedProx parameter mapping OK ({len(_REFERENCE_KEYS)} tensors).")
del _test_model


## 3. Evaluation Helper (shared by server-side eval and final test-set eval)

In [ ]:
def evaluate_model(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(next(model.parameters()).device) for k, v in batch.items()}
            out = model(**batch)
            total_loss += out.loss.item() * batch["labels"].size(0)
            all_preds.extend(out.logits.argmax(dim=-1).cpu().numpy())
            all_labels.extend(batch["labels"].cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    _, _, f1_macro, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    _, _, f1_weighted, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted", zero_division=0)
    return {
        "loss": total_loss / len(loader.dataset), "accuracy": acc,
        "f1_macro": f1_macro, "f1_weighted": f1_weighted,
        "preds": all_preds, "labels": all_labels,
    }

## 4. Flower Client

Each simulated hospital is one client. `fit()` optionally adds the FedProx
proximal term (controlled via `config["proximal_mu"]`, sent by the server each
round) — when μ=0 this is exactly plain FedAvg local training. The proximal term uses live
parameters (not detached state-dict copies), so it genuinely affects training.

In [ ]:
class HospitalClient(NumPyClient):
    def __init__(self, hospital_idx, data_source):
        self.hospital_idx = hospital_idx
        train_df = load_hospital_split(hospital_idx, "train", data_source)
        train_df["label_id"] = train_df["syndrome_category"].map(label2id)
        self.train_loader = DataLoader(
            SymptomDataset(train_df, tokenizer), batch_size=BATCH_SIZE, shuffle=True
        )
        self.n_train = len(train_df)
        # Pick the device inside the Ray actor (the notebook-level device may not match)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = build_lora_model().to(self.device)
        self.prox_params = get_prox_params(self.model)

    def get_parameters(self, config):
        return get_lora_ndarrays(self.model)

    def fit(self, parameters, config):
        set_lora_ndarrays(self.model, parameters)
        mu = config.get("proximal_mu", 0.0)
        # Frozen snapshot of the GLOBAL params received this round, used as the
        # anchor point for the FedProx proximal term (no grad needed on these).
        global_tensors = [torch.tensor(p).to(self.device) for p in parameters]

        optimizer = torch.optim.AdamW(self.model.parameters(), lr=LOCAL_LR)
        self.model.train()
        last_loss = 0.0
        for _ in range(LOCAL_EPOCHS):
            for batch in self.train_loader:
                batch = {k: v.to(self.device) for k, v in batch.items()}
                optimizer.zero_grad()
                out = self.model(**batch)
                loss = out.loss
                if mu > 0:
                    prox_term = sum(
                        torch.sum((p - g) ** 2)
                        for p, g in zip(self.prox_params, global_tensors)
                    )
                    loss = loss + (mu / 2.0) * prox_term
                loss.backward()
                optimizer.step()
                last_loss = loss.item()

        return get_lora_ndarrays(self.model), self.n_train, {"train_loss": float(last_loss)}

    def evaluate(self, parameters, config):
        set_lora_ndarrays(self.model, parameters)
        self.model.to(self.device)
        metrics = evaluate_model(self.model, self.train_loader)  # quick local check, not used for reported results
        return float(metrics["loss"]), self.n_train, {"accuracy": float(metrics["accuracy"])}

## 5. Server-Side Evaluation + Simulation Runner

Server-side `evaluate_fn` runs the aggregated GLOBAL model on the pooled
validation set after every round — this is what produces the convergence
curves, and keeps the test set untouched until the very end.

In [ ]:
def run_inprocess_experiment(run_name, data_source, proximal_mu, num_rounds=NUM_ROUNDS):
    """Same FedAvg/FedProx maths as the Flower run, executed sequentially without Ray."""
    import gc
    print(f"\n{'='*60}\nRun (in-process): {run_name} | data={data_source} | proximal_mu={proximal_mu}\n{'='*60}")
    pooled_val_loader, pooled_test_loader = pooled_loaders[data_source]
    global_params = get_lora_ndarrays(build_lora_model())
    init_arrs = [a.copy() for a in global_params]
    eval_model = build_lora_model().to(DEVICE)
    round_history = []

    def eval_global(rnd):
        set_lora_ndarrays(eval_model, global_params)
        m = evaluate_model(eval_model, pooled_val_loader)
        round_history.append({"round": rnd, "val_loss": m["loss"],
                              "val_accuracy": m["accuracy"], "val_f1_macro": m["f1_macro"]})
        print(f"  [round {rnd}] val_loss={m['loss']:.4f} val_acc={m['accuracy']:.4f} val_f1_macro={m['f1_macro']:.4f}")

    eval_global(0)
    start = time.time()
    for rnd in range(1, num_rounds + 1):
        updates, sizes = [], []
        for h in range(NUM_HOSPITALS):
            client = HospitalClient(h, data_source)          # fresh model each round, then freed
            params, n, _ = client.fit(global_params, {"proximal_mu": proximal_mu})
            updates.append(params); sizes.append(n)
            del client; gc.collect(); torch.cuda.empty_cache()
        total = float(sum(sizes))
        global_params = [sum(u[i] * (n / total) for u, n in zip(updates, sizes)) for i in range(len(global_params))]
        eval_global(rnd)
    elapsed = time.time() - start
    print(f"Run '{run_name}' finished in {elapsed:.1f}s over {num_rounds} rounds.")

    test_metrics = evaluate_model(eval_model, pooled_test_loader)
    print(f"Final TEST metrics for '{run_name}': acc={test_metrics['accuracy']:.4f} f1_macro={test_metrics['f1_macro']:.4f}")
    per_round_mb = transmitted_size_mb(init_arrs) * NUM_HOSPITALS * 2
    return {"run_name": run_name, "data_source": data_source, "proximal_mu": proximal_mu,
            "round_history": round_history, "test_metrics": test_metrics,
            "elapsed_sec": elapsed, "total_comm_mb": per_round_mb * num_rounds}


def run_federated_experiment(run_name, data_source, proximal_mu, num_rounds=NUM_ROUNDS):
    if SIMULATION_BACKEND == "inprocess":
        return run_inprocess_experiment(run_name, data_source, proximal_mu, num_rounds)
    print(f"\n{'='*60}\nRun: {run_name} | data={data_source} | proximal_mu={proximal_mu}\n{'='*60}")

    pooled_val_loader, pooled_test_loader = pooled_loaders[data_source]
    round_history = []  # captured via closure inside evaluate_fn below
    eval_model_holder = {"model": None}  # lazily built once, reused across rounds
    init_arrs = get_lora_ndarrays(build_lora_model())  # computed once here, reused below AND inside server_fn

    def evaluate_fn(server_round, parameters, config):
        if eval_model_holder["model"] is None:
            eval_model_holder["model"] = build_lora_model().to(DEVICE)
        set_lora_ndarrays(eval_model_holder["model"], parameters)
        metrics = evaluate_model(eval_model_holder["model"], pooled_val_loader)
        round_history.append({
            "round": server_round, "val_loss": metrics["loss"],
            "val_accuracy": metrics["accuracy"], "val_f1_macro": metrics["f1_macro"],
        })
        print(f"  [round {server_round}] val_loss={metrics['loss']:.4f} "
              f"val_acc={metrics['accuracy']:.4f} val_f1_macro={metrics['f1_macro']:.4f}")
        return metrics["loss"], {"accuracy": metrics["accuracy"], "f1_macro": metrics["f1_macro"]}

    def client_fn(context: Context):
        hospital_idx = context.node_config["partition-id"]
        return HospitalClient(hospital_idx, data_source).to_client()

    def server_fn(context: Context):
        strategy = FedAvg(
            fraction_fit=1.0, fraction_evaluate=0.0,   # server-side evaluate_fn used instead of client-side eval
            min_fit_clients=NUM_HOSPITALS, min_available_clients=NUM_HOSPITALS,
            initial_parameters=ndarrays_to_parameters(init_arrs),
            evaluate_fn=evaluate_fn,
            on_fit_config_fn=lambda rnd: {"proximal_mu": proximal_mu},
        )
        return ServerAppComponents(strategy=strategy, config=ServerConfig(num_rounds=num_rounds))

    client_app = ClientApp(client_fn=client_fn)
    server_app = ServerApp(server_fn=server_fn)

    start = time.time()
    run_simulation(
        server_app=server_app, client_app=client_app, num_supernodes=NUM_HOSPITALS,
        backend_config={"client_resources": CLIENT_RESOURCES},
    )
    elapsed = time.time() - start
    print(f"Run '{run_name}' finished in {elapsed:.1f}s over {num_rounds} rounds.")
    if len({round(h["val_loss"], 6) for h in round_history}) == 1:
        raise RuntimeError("Global model never changed: all client updates failed. Set SIMULATION_BACKEND = \"inprocess\".")

    # Final global model (last round's parameters, held in eval_model_holder) -> test set
    test_metrics = evaluate_model(eval_model_holder["model"], pooled_test_loader)
    print(f"Final TEST metrics for '{run_name}': acc={test_metrics['accuracy']:.4f} "
          f"f1_macro={test_metrics['f1_macro']:.4f}")

    # Communication cost estimate
    per_round_mb = transmitted_size_mb(init_arrs) * NUM_HOSPITALS * 2  # up + down per client
    total_comm_mb = per_round_mb * num_rounds

    return {
        "run_name": run_name, "data_source": data_source, "proximal_mu": proximal_mu,
        "round_history": round_history, "test_metrics": test_metrics,
        "elapsed_sec": elapsed, "total_comm_mb": total_comm_mb,
    }

## 6. Run All Four Experiments

This is the long-running cell. Rough estimate on a T4: each round trains all three
hospitals for one local epoch (~10.5k rows in total, about the cost of one Phase 3
epoch, ~2.5 min), so expect roughly 15-20 minutes per run and about 1-1.5 hours for
all four. To sanity-check first, set `NUM_ROUNDS = 1` in the config cell. If Colab
disconnects, finished results are lost, so consider running one experiment per cell.


In [ ]:
experiment_configs = [
    ("FedAvg_nonIID",     "non_iid", 0.0),
    ("FedProx_mu0.01_nonIID", "non_iid", 0.01),
    ("FedProx_mu0.1_nonIID",  "non_iid", 0.1),
    ("FedAvg_IID",        "iid",     0.0),
]

all_results = {}
for run_name, data_source, mu in experiment_configs:
    all_results[run_name] = run_federated_experiment(run_name, data_source, mu)

# Persist raw results (minus in-memory model objects) for later reuse
def _to_py(o):
    # preds/labels hold numpy ints, which json can't serialise by default
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    raise TypeError(f"Not serialisable: {type(o)}")

with open(os.path.join(OUTPUT_DIR, "federated_results.json"), "w") as f:
    json.dump(all_results, f, indent=2, default=_to_py)
print("\nAll experiments complete. Results saved.")

## 7. Convergence Curves — All Strategies Overlaid

In [ ]:
plt.figure(figsize=(9, 5))
for run_name, r in all_results.items():
    hist_df = pd.DataFrame(r["round_history"])
    plt.plot(hist_df["round"], hist_df["val_f1_macro"], marker="o", label=run_name)

plt.axhline(y=CENTRALIZED_BASELINE["f1_macro"], color="black", linestyle="--",
            label=f"Centralized, no augment ({CENTRALIZED_BASELINE['f1_macro']:.3f})")
plt.axhline(y=CENTRALIZED_BASELINE_AUG["f1_macro"], color="gray", linestyle=":",
            label=f"Centralized, augmented ({CENTRALIZED_BASELINE_AUG['f1_macro']:.3f})")
plt.xlabel("Communication Round")
plt.ylabel("Validation F1 (macro)")
plt.title("Federated Convergence: FedAvg vs FedProx, IID vs Non-IID")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "convergence_curves.png"), dpi=150)
plt.show()

## 8. Final Comparison Table

In [ ]:
rows = []
for run_name, r in all_results.items():
    tm = r["test_metrics"]
    rows.append({
        "run": run_name,
        "data_split": r["data_source"],
        "proximal_mu": r["proximal_mu"],
        "test_accuracy": round(tm["accuracy"], 4),
        "test_f1_macro": round(tm["f1_macro"], 4),
        "test_f1_weighted": round(tm["f1_weighted"], 4),
        "gap_vs_central_noaug_f1": round(CENTRALIZED_BASELINE["f1_macro"] - tm["f1_macro"], 4),
        "gap_vs_central_aug_f1": round(CENTRALIZED_BASELINE_AUG["f1_macro"] - tm["f1_macro"], 4),
        "wall_time_sec": round(r["elapsed_sec"], 1),
        "total_comm_mb": round(r["total_comm_mb"], 2),
    })

comparison_df = pd.DataFrame(rows).sort_values("test_f1_macro", ascending=False)
print(comparison_df.to_string(index=False))
comparison_df.to_csv(os.path.join(OUTPUT_DIR, "federated_comparison.csv"), index=False)

print(f"\nCentralized baselines (Phase 3): "
      f"no-augment acc={CENTRALIZED_BASELINE['accuracy']:.4f} f1={CENTRALIZED_BASELINE['f1_macro']:.4f} | "
      f"augmented acc={CENTRALIZED_BASELINE_AUG['accuracy']:.4f} f1={CENTRALIZED_BASELINE_AUG['f1_macro']:.4f}")
print("Note: the IID run is scored on the IID pooled test set, the others on the non-IID one (same as Phase 3).")
print("\nInterpretation notes:")
print("- Compare FedAvg_nonIID vs FedAvg_IID: the gap between them is the real")
print("  cost of non-IID data specifically (holding the aggregation algorithm fixed).")
print("- Compare FedAvg_nonIID vs the FedProx runs: if FedProx converges faster")
print("  or reaches higher F1 under non-IID, that's evidence the proximal term is")
print("  doing its job of limiting client drift.")
print("- 'gap_vs_central_noaug_f1' is the cost of federating at all vs. having all data in one")
print("  place, like-for-like (neither side augments). A negative gap means federated did better.")

## 9. Confusion Matrix — Best Federated Run

In [ ]:
best_run_name = comparison_df.iloc[0]["run"]
best_run = all_results[best_run_name]
tm = best_run["test_metrics"]
print(f"Best federated run: {best_run_name}")
print(classification_report(tm["labels"], tm["preds"], target_names=CATEGORIES, zero_division=0))

cm = confusion_matrix(tm["labels"], tm["preds"])
plt.figure(figsize=(8, 6.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=CATEGORIES, yticklabels=CATEGORIES)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title(f"Confusion Matrix — Best Federated Run ({best_run_name})")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "federated_confusion_matrix.png"), dpi=150)
plt.show()

## 10. What to Look For in Your Results

- **FedAvg_IID should be your best or near-best result and should converge
  fastest.** If it isn't, something about the setup (learning rate, local
  epochs) may need tuning independent of the non-IID question — diagnose that
  before drawing conclusions about non-IID/FedProx effects.
- **FedAvg_nonIID vs. FedAvg_IID gap** is your headline "cost of non-IID data"
  finding for the report.
- **Whether FedProx actually helps** (either μ) over plain FedAvg on the
  non-IID split is genuinely an open empirical question for your specific
  data — don't assume it must win; report whichever way it actually goes,
  since a null result here ("FedProx didn't meaningfully help at this scale")
  is still a valid, reportable finding, especially with only 3 clients where
  client drift has less room to compound than in larger federations.
- **`gap_vs_central_noaug_f1`** across all four runs tells you the overall
  federation cost. Compare against the no-augmentation centralized run (0.913),
  since federated clients don't augment. A small gap, or even a slightly negative
  one, is a good outcome and shows federation approximates centralized performance
  while keeping data local.
- **Differences of a few tenths of a point are within noise.** Each run uses one
  seed and a ~2,250-row test set (about +/-0.5 points of sampling error), so don't
  rank runs that differ by less than that. A FedProx vs FedAvg gap this small is a
  null result.
- **Likely weak class:** `Musculoskeletal/Other` is the catch-all category and was the
  hardest class in Phase 3 (recall 0.82); expect the same here. Under non-IID splits
  it may suffer more if one hospital holds most of its examples.
- **Caveat for the report:** numbers come from template-generated text and keyword-rule
  labels, so they show relative differences between FL strategies, not real-world accuracy.